<a href="https://colab.research.google.com/github/AvantiShri/gotpsi_sequentialcard/blob/main/notebooks/GotPsi_SequentialCard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Get the data

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls /content/drive/MyDrive/IONS/Dean_SequentialCard

 cardS16.zip  'Dean SequentialCard README.gdoc'  'raw cardsequence.zip'


In [ ]:
!md5sum "/content/drive/MyDrive/IONS/Dean_SequentialCard/cardS16.zip" #md5 checksum to compare integrity of file

29899584767d64ec6d3cfcd7496d751a  /content/drive/MyDrive/IONS/Dean_SequentialCard/cardS16.zip


In [3]:
!unzip -q "/content/drive/MyDrive/IONS/Dean_SequentialCard/cardS16.zip" -d /content/cardsequence #takes about a minute

In [4]:
#The data for the first two quarters of 2002 is hiding in cardS02Q1.zip and cardS02Q2.zip, so we will further
# unzip those
!unzip -q /content/cardsequence/cardS02/cardS02Q1.zip -d /content/cardsequence/cardS02
!unzip -q /content/cardsequence/cardS02/cardS02Q2.zip -d /content/cardsequence/cardS02

# Install dependencies

In [16]:
! pip uninstall -y gotpsi_sequentialcard
! pip install git+https://github.com/AvantiShri/gotpsi_sequentialcard.git

Found existing installation: gotpsi_sequentialcard 0.1.0
Uninstalling gotpsi_sequentialcard-0.1.0:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/gotpsi_sequentialcard-0.1.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/gotpsi_sequentialcard/*
Proceed (Y/n)? Y
  Successfully uninstalled gotpsi_sequentialcard-0.1.0
  Cloning https://github.com/AvantiShri/gotpsi_sequentialcard.git to /tmp/pip-req-build-ji85_v01
  Running command git clone --filter=blob:none --quiet https://github.com/AvantiShri/gotpsi_sequentialcard.git /tmp/pip-req-build-ji85_v01
  Resolved https://github.com/AvantiShri/gotpsi_sequentialcard.git to commit cf1d2267f114c94934f5800d2da34b0446b850c4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for gotpsi_sequentialcard: filename=gotpsi_sequentialcard-0.1.0-py3-none-any.whl size=2822 sha256=daa7dfe91306e46d608938286342450b782cd4f6dd9eaf88fc98e2cdb1

In [17]:
from importlib import reload
import gotpsi_sequentialcard.utils
reload(gotpsi_sequentialcard.utils)

<module 'gotpsi_sequentialcard.utils' from '/usr/local/lib/python3.12/dist-packages/gotpsi_sequentialcard/utils.py'>

In [18]:
from gotpsi_sequentialcard.utils import find_missing_dates

missing_ranges, earliest_date, latest_date, available_dates = find_missing_dates("/content/cardsequence")

print(f"Data spans: {earliest_date} to {latest_date}")
print(f"\n{len(missing_ranges)} missing date range(s):\n")

for start, end in missing_ranges:
    if start == end:
        print(f"  {start}")
    else:
        days = (end - start).days + 1
        print(f"  {start} to {end}  ({days} days)")

Data spans: 2002-01-01 to 2018-12-29

6 missing date range(s):

  2015-08-13 to 2015-09-20  (39 days)
  2015-09-24 to 2015-09-27  (4 days)
  2015-09-29 to 2015-10-03  (5 days)
  2015-10-07 to 2015-10-09  (3 days)
  2015-10-11 to 2015-10-26  (16 days)
  2015-12-04 to 2016-05-12  (161 days)


In [34]:
from collections import namedtuple
from datetime import datetime
import os

Trial = namedtuple('Trial', ['userid', 'trial', 'target_card', 'click_sequence', 'image', 'timestamp'])

# Completed trial format:
#   userid, trial, steps, r1,r2,r3,r4,r5,r6, imagepath, timestamp
# Example: rada, 1, 2, 0,5,4,0,0,0, ../../bi/images4/c9.jpg, Thu Jan  1 00:05:52 2009
def parse_trial_file(filepath):
    trials = []

    with open(filepath, 'r') as f:
        for line_num, line in enumerate(f, 1):
            if 'image' not in line:
                continue

            try:
                parts = [p.strip() for p in line.split(',')]
                assert len(parts) == 11, f"Expected 11 parts, got {len(parts)}"

                userid = parts[0]
                trial_num = int(parts[1])
                steps = int(parts[2])
                clicks = [int(parts[i]) for i in range(3, 9)]
                imagepath = parts[9]
                timestamp_str = parts[10].strip()
                timestamp = datetime.strptime(timestamp_str, '%a %b %d %H:%M:%S %Y')

                image = os.path.basename(imagepath)

                assert clicks[0] == 0, (
                    f"Expected r1=0, got {clicks[0]}"
                )

                trailing = clicks[steps + 1:]
                assert len(trailing) == 0 or all(r == 0 for r in trailing), (
                    f"Expected trailing zeros after step {steps}, got {clicks}"
                )

                click_sequence = clicks[1:steps + 1]
                target_card = click_sequence[-1]

                trials.append(Trial(userid, trial_num, target_card, click_sequence, image, timestamp))

            except Exception as e:
                print(f"Error in {filepath}, line {line_num}: {e}")
                print(f"  Content: {line.rstrip()}")

    return trials

In [35]:
#from gotpsi_sequentialcard.utils import parse_trial_file

parsed_trials = parse_trial_file("/content/cardsequence/cardS02/cardS020202.dat")

Error in /content/cardsequence/cardS02/cardS020202.dat, line 246: invalid literal for int() with base 10: ''
  Content: shadow drakken, 15, 5, 0,4,3,,5,1, ../../bi/images4/ok.jpg, Sat Feb  2 00:28:03 2002
Error in /content/cardsequence/cardS02/cardS020202.dat, line 255: invalid literal for int() with base 10: ''
  Content: shadow drakken, 17, 5, 0,4,2,3,,5, ../../bi/images4/c8.jpg, Sat Feb  2 00:28:30 2002
Error in /content/cardsequence/cardS02/cardS020202.dat, line 888: Expected 11 parts, got 12
  Content: igryphon, 1, 6, 0,,2,4,1,5,3, ../../bi/images4/ok.jpg, Sat Feb  2 04:25:53 2002
Error in /content/cardsequence/cardS02/cardS020202.dat, line 899: invalid literal for int() with base 10: ''
  Content: igryphon, 4, 3, 0,,4,1,0,0, ../../bi/images4/c4.gif, Sat Feb  2 04:26:14 2002
Error in /content/cardsequence/cardS02/cardS020202.dat, line 6555: invalid literal for int() with base 10: ''
  Content: frofnuss, 11, 4, 0,2,,4,5,0, ../../bi/images4/c8.gif, Sat Feb  2 14:05:17 2002
Error in 

In [42]:
max([x.trial for x in parsed_trials])

25

In [44]:
parsed_trials2 = parse_trial_file("/content/cardsequence/cardS18/cardS180101.dat")

In [46]:
max([x.trial for x in parsed_trials2])

25

In [52]:
[x for x in parsed_trials2 if x.userid=="ovalpool"]

[Trial(userid='ovalpool', trial=1, target_card=3, click_sequence=[3], image='c6.jpg', timestamp=datetime.datetime(2018, 1, 1, 2, 16, 44)),
 Trial(userid='ovalpool', trial=2, target_card=2, click_sequence=[4, 3, 2], image='c8.gif', timestamp=datetime.datetime(2018, 1, 1, 2, 16, 48)),
 Trial(userid='ovalpool', trial=3, target_card=4, click_sequence=[4], image='c9.jpg', timestamp=datetime.datetime(2018, 1, 1, 2, 16, 50)),
 Trial(userid='ovalpool', trial=4, target_card=3, click_sequence=[4, 5, 3], image='ok.jpg', timestamp=datetime.datetime(2018, 1, 1, 2, 16, 54)),
 Trial(userid='ovalpool', trial=5, target_card=4, click_sequence=[5, 3, 2, 1, 4], image='c7.gif', timestamp=datetime.datetime(2018, 1, 1, 2, 17)),
 Trial(userid='ovalpool', trial=6, target_card=3, click_sequence=[3], image='ok.jpg', timestamp=datetime.datetime(2018, 1, 1, 2, 17, 3)),
 Trial(userid='ovalpool', trial=7, target_card=1, click_sequence=[5, 3, 2, 1], image='ok.jpg', timestamp=datetime.datetime(2018, 1, 1, 2, 17, 8)),


In [49]:
[x for x in parsed_trials2 if x.userid=="rada"]

[Trial(userid='rada', trial=1, target_card=3, click_sequence=[1, 3], image='ok.jpg', timestamp=datetime.datetime(2018, 1, 1, 1, 7, 7)),
 Trial(userid='rada', trial=2, target_card=2, click_sequence=[1, 4, 2], image='c5.gif', timestamp=datetime.datetime(2018, 1, 1, 1, 7, 13)),
 Trial(userid='rada', trial=3, target_card=2, click_sequence=[1, 3, 5, 2], image='c6.gif', timestamp=datetime.datetime(2018, 1, 1, 1, 7, 20)),
 Trial(userid='rada', trial=4, target_card=5, click_sequence=[2, 4, 5], image='ok.jpg', timestamp=datetime.datetime(2018, 1, 1, 1, 7, 25)),
 Trial(userid='rada', trial=5, target_card=5, click_sequence=[2, 1, 4, 3, 5], image='c6.gif', timestamp=datetime.datetime(2018, 1, 1, 1, 7, 35)),
 Trial(userid='rada', trial=6, target_card=4, click_sequence=[5, 2, 3, 1, 4], image='c9.gif', timestamp=datetime.datetime(2018, 1, 1, 1, 7, 41)),
 Trial(userid='rada', trial=7, target_card=2, click_sequence=[4, 2], image='c4.gif', timestamp=datetime.datetime(2018, 1, 1, 1, 7, 45)),
 Trial(useri

In [39]:
!cat cardsequence/cardS02/cardS020202.dat | grep 'chrisreed'

chrisreed, 1, 2, Sat Feb  2 16:45:36 2002
chrisreed, 1, 3, Sat Feb  2 16:45:40 2002
chrisreed, 1, 2, 0,2,3,0,0,0, ../../bi/images4/c5.gif, Sat Feb  2 16:45:40 2002
chrisreed, 2, 2, Sat Feb  2 16:45:46 2002
chrisreed, 2, 4, Sat Feb  2 16:45:48 2002
chrisreed, 2, 3, Sat Feb  2 16:45:49 2002
chrisreed, 2, 3, 0,2,4,3,0,0, ../../bi/images4/c3.gif, Sat Feb  2 16:45:49 2002
chrisreed, 3, 5, Sat Feb  2 16:45:53 2002
chrisreed, 3, 1, 0,5,0,0,0,0, ../../bi/images4/c3.gif, Sat Feb  2 16:45:53 2002
chrisreed, 4, 2, Sat Feb  2 16:45:56 2002
chrisreed, 4, 1, 0,2,0,0,0,0, ../../bi/images4/ok.jpg, Sat Feb  2 16:45:56 2002
chrisreed, 5, 4, Sat Feb  2 16:46:05 2002
chrisreed, 5, 3, Sat Feb  2 16:46:07 2002
chrisreed, 5, 1, Sat Feb  2 16:46:08 2002
chrisreed, 5, 3, 0,4,3,1,0,0, ../../bi/images4/c3.jpg, Sat Feb  2 16:46:08 2002
chrisreed, 6, 2, Sat Feb  2 16:46:13 2002
chrisreed, 6, 5, Sat Feb  2 16:46:14 2002
chrisreed, 6, 2, 0,2,5,0,0,0, ../../bi/images4/ok.jpg, Sat Feb  2 16:46:14 2002
chrisreed, 7, 4,